# Modèle prédictif - Élections présidentielles Île-de-France

Objectif : prédire la **part de vote** (part du gagnant ou d'une famille politique) à partir des indicateurs socio-économiques et de l'historique électoral.

- **Périmètre** : départements IDF (75, 77, 78, 91, 92, 93, 94, 95)
- **Données** : 1969–2022, 1er tour
- **Split** : temporel (train sur les années passées, test sur 2017 et 2022)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

from src.ml.data import build_ml_dataset, ELECTION_YEARS
from src.ml.train import get_feature_columns, prepare_xy, temporal_split, train_and_evaluate
import pandas as pd

In [2]:
# Chargement du jeu de données (depuis la base ou l'ETL)
# use_db=False : toutes les années 1969-2022 via l'ETL. use_db=True : BDD (si partielle, peu d'années)
df = build_ml_dataset(target="share_winner", use_db=False, include_lags=True)
print(f"Lignes : {len(df)}, Colonnes : {list(df.columns)}")
print(f"Années présentes : {sorted(df['year'].unique())}")
df.sort_values(["year", "dept_code"]).head(20)

Lignes : 80, Colonnes : ['year', 'dept_code', 'autre', 'centre', 'divers', 'droite', 'droite_nat', 'extreme_droite', 'extreme_gauche', 'gauche', 'share_winner', 'median_standard_of_living', 'no_diploma_rate_20_24', 'poverty_rate', 'social_housing_share', 'turnout_rate', 'unemployment_rate', 'share_winner_prev', 'extreme_gauche_prev', 'gauche_prev', 'centre_prev', 'droite_prev', 'extreme_droite_prev', 'droite_nat_prev', 'autre_prev', 'election_number', 'target']


/home/aschwarz/Bureau/Personnel/MasterEPSI/MSPR/ELECTION/mspr/src/ml/data.py:82: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  results_df = pd.read_sql(results_sql, conn)


,year,dept_code,autre,centre,divers,droite,droite_nat,extreme_droite,extreme_gauche,gauche,...,share_winner_prev,extreme_gauche_prev,gauche_prev,centre_prev,droite_prev,extreme_droite_prev,droite_nat_prev,autre_prev,election_number,target
0,1969,75,0.5478,0.0,0.0,0.4523,0.0,0.000,0.000,0.000,...,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,0.0000,1,0.4523
1,1969,77,0.5570,0.0,0.0,0.4431,0.0,0.000,0.000,0.000,...,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,0.0000,1,0.4431
2,1969,78,0.5638,0.0,0.0,0.4361,0.0,0.000,0.000,0.000,...,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,0.0000,1,0.4361
3,1969,91,0.6042,0.0,0.0,0.3957,0.0,0.000,0.000,0.000,...,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,0.0000,1,0.3957
4,1969,92,0.5837,0.0,0.0,0.4162,0.0,0.000,0.000,0.000,...,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,0.0000,1,0.4162
5,1969,93,0.6603,0.0,0.0,0.3397,0.0,0.000,0.000,0.000,...,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,0.0000,1,0.3863
6,1969,94,0.6151,0.0,0.0,0.3849,0.0,0.000,0.000,0.000,...,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,0.0000,1,0.3849
7,1969,95,0.6066,0.0,0.0,0.3934,0.0,0.000,0.000,0.000,...,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,0.0000,1,0.3934
8,1974,75,0.6000,0.0,0.0,0.0000,0.0,0.009,0.016,0.373,...,0.4523,0.0,0.0,0.0,0.4523,0.0,0.0,0.5478,2,0.3950
9,1974,77,0.5380,0.0,0.0,0.0000,0.0,0.009,0.026,0.428,...,0.4431,0.0,0.0,0.0,0.4431,0.0,0.0,0.5570,2,0.4280


In [ ]:
# Répartition train / test (temporelle)
train_df, test_df = temporal_split(df, test_years=[2017, 2022])
print(f"Train : {len(train_df)} lignes, Test : {len(test_df)} lignes")
print(f"Features utilisées : {get_feature_columns()}")

In [ ]:
# Entraînement et évaluation
metrics = train_and_evaluate(
    target="share_winner",
    use_db=True,
    test_years=[2017, 2022],
    output_dir=Path("../data/processed/ml"),
    model_type="ridge",
)
pd.Series(metrics)

## Interprétation

- **MAE** : erreur absolue moyenne sur la part de vote (ex: 0.05 = 5 points)
- **R²** : part de variance expliquée
- Le modèle (Ridge ou Random Forest) peut être relancé en ligne de commande :
  `python -m src.ml.train --target share_winner --model ridge`